# Stage 10: Pseudobulk Differential Expression and Positive-Control Evaluation

> **Outputs read: the active config** (printed by the first code cell). The default since 2026-11-15 is **refs5**, the
> registered rerun of Stages 8–14 after the 2026-11-11 audit (`docs/audit/remediation/STEP3_RESULTS.md`): the 15
> references rebuilt into `references_v5/` (HYPOTH re-annotated, the HEART merge applied), the Stage 8 exclusion list
> rebuilt in Ensembl 115, the target sum set by the registered formula, and the repaired gate, DE and network code.
> **Model: retrain v2** (phase 2 final). The default before refs5 (`_refs4all_rebuild` folders) is
> `config/profiles/refs4all_rebuild.yaml`; this notebook as executed on it is in git history at commit `12f94b9`
> (the 14-tissue refs4 version at `542b454`). Earlier comparisons: `deconvolution/ALL_TISSUE_RERUN_RESULTS.md`
> (refs4 against all tissues), `deconvolution/REFS4_RERUN_RESULTS.md` (July against refs4) and
> `docs/RETRAIN_V2_RESULTS.md` (April against v2).
> The DE does not use the model; v2 enters through the hotspot flags.

## Stage Objective

Quantify exercise-associated transcriptional changes by cell type and evaluate recovery of pre-registered biological positive controls.

## Workflow

| Step | Operation | Analytical purpose |
|------|-----------|--------------------|
| 1 | Pseudobulk assembly | Aggregate cell-type-resolved expression at the sample level. |
| 2 | Differential modeling | Fit limma-trend within each sex, combine the sexes present, and apply global false-discovery-rate control. |
| 3 | Hotspot flags | Copy each block's class from the Stage 11 gate (hotspot, borderline or none) into the DE tables. |
| 4 | Positive-control comparison | Compare detected genes and pathways with the frozen Tier A reference set. |

## Inputs and Outputs

| Component | Description |
|-----------|-------------|
| Inputs | Stage 8 pseudo-cell expression and sample metadata, plus the pre-registered positive-control specification. |
| Primary outputs | Differential-expression tables, hotspot summaries, and positive-control comparison reports. |

## Result Overview

The analysis reports both recovered and unrecovered controls, avoiding interpretation based only on significant findings.

The sections below present the computational evidence in execution order. Every figure is printed by a code cell from the recorded stage outputs or the config; the markdown describes methods and points to those cells rather than quoting numbers.

In [1]:
from pathlib import Path

from nb_config import DE_DIR
from nb_config import TISSUES as tissues

print("=== Global outputs ===")
for f in ["de_summary.tsv","de_hotspots.tsv","de_methods.tsv",
          "posctrl_results.tsv","posctrl_responsiveness.tsv","posctrl_summary.md",
          "dominant_celltype_flags.tsv"]:
    p = DE_DIR / f
    size = f"{p.stat().st_size/1e3:.1f} KB" if p.exists() else "MISSING"
    rows = sum(1 for _ in p.open()) - 1 if p.exists() and f.endswith(".tsv") else ""
    print(f"  {f:<40} {size:>12}  {str(rows)+' rows' if rows else ''}")

print()
print("=== Per-tissue DE files ===")
for t in tissues:
    d = DE_DIR / t
    n = len(list(d.glob("de__*.tsv"))) if d.exists() else 0
    total_kb = sum(f.stat().st_size for f in d.glob("de__*.tsv")) / 1e3 if d.exists() else 0
    print(f"  {t:<10}  {n:>3} cell-type DE files  ({total_kb:>7.0f} KB total)")


config: (default config)
Stage 8: /depot/reese18/apps/motrpac-genecompass/data/deconvolution/results_refs4all_rebuild/motrpac
Stage 9: /depot/reese18/apps/motrpac-genecompass/data/deconvolution/genecompass_input_v2_refs4all_rebuild
16 tissues: BAT, BLOOD, CORTEX, HEART, HIPPOC, HYPOTH, KIDNEY, LIVER, LUNG, OVARY, SKMGN, SKMVL, SMLINT, SPLEEN, TESTES, WATSC
=== Global outputs ===
  de_summary.tsv                                32.1 KB  197 rows
  de_hotspots.tsv                                1.9 KB  9 rows
  de_methods.tsv                                 0.7 KB  10 rows
  posctrl_results.tsv                           15.4 KB  81 rows
  posctrl_responsiveness.tsv                     1.6 KB  11 rows
  posctrl_summary.md                            10.8 KB  
  dominant_celltype_flags.tsv                    0.8 KB  16 rows

=== Per-tissue DE files ===
  BAT           6 cell-type DE files  (  43931 KB total)
  BLOOD         8 cell-type DE files  (  59635 KB total)
  CORTEX       10 cell-type

## Step 1 — Per-cell-type Pseudobulk DE (limma-trend)

`run_pseudobulk_de.sh` runs `deconvolution/R/run_pseudobulk_de.R`, an adaptation of the MoTrPAC / Vetr 2024 pseudobulk
DE recipe to BayesPrism posterior Z. For each (tissue, cell type) block independently:

1. **Input**: `pred_z/predz__<CellType>.csv` — samples × genes, continuous count-scale Z
2. **Filter**: drop only all-zero genes (every gene expressed in at least one sample is tested)
3. **Gate**: a block is analysed only if its median per-sample library is at least the §6 threshold
   (`REFERENCE_RELABEL_PREREG.md` §6); the others are skipped as near-empty and recorded in `de_summary.tsv`
4. **Scale**: log2-CPM (`edgeR::cpm`, prior count 1) of Z, fitted with limma `eBayes(trend = TRUE, robust = TRUE)`;
   Z is continuous, so a count model on rounded values would zero out the rare cell types
5. **Models** (`de_methods.tsv` records them for each run):
   - combined, both sexes: `~ sex + factor(week)` (omnibus F over the training weeks and per-week contrasts vs
     control), `~ sex + week` (ordinal dose slope), `~ sex * factor(week)` (sex × dose interaction). A one-sex block
     drops the sex term, and the interaction is fitted in two-sex blocks only. These fits are descriptive
   - per sex: `~ factor(week)` within each sex present (per-sex omnibus and per-week contrasts). This is the dose test
6. **Sex-combine**: Fisher's combination of the per-sex omnibus p over the sexes present: chi-square on 4 degrees of
   freedom with two sexes, and the one sex's own p in a single-sex tissue (OVARY, TESTES). Per-block BH is kept as a
   secondary. Before refs5 the combined p was missing whenever a sex was absent, so the single-sex tissues could not
   have a dose-significant gene (audit finding 7.1)
7. **Global FDR**: IHW over the Fisher p of every analysed gene-block, with tissue as a nominal covariate. Before
   refs5 tissue was passed with IHW's default covariate type, ordinal, which smoothed the weights across
   alphabetically adjacent tissues (audit finding 7.8)
8. **Sex-consistency at 8w**: the primary reading is concordance: both per-sex 8-week tests significant (BH within
   the block) with the same sign. The secondary reading is repfdr, fitted on the dose-significant gene-blocks only
   (FDR_IHW < 0.05), as the consortium does; a call is a tail-area Fdr below 0.05 with the same sign.
   `de_summary.tsv` carries both counts and their union. Before refs5 repfdr was fitted on every gene-block, fitted
   poorly, and supplied nearly all of the calls (audit findings 7.2 and 7.9)

**Hotspot criterion**: a block is flagged `is_hotspot=TRUE` when its class in the Aim-2 gate table is `hotspot`
(`gate_class`, copied into `de_summary.tsv`; `deconvolution.hotspots_table` in the config). The class is a
Benjamini-Hochberg call over the analysable blocks on the permutation p of the supervised PLS-1 probe (trained vs
control on the v2 embeddings, Stage 11, 10 stratified 5-fold splits) that more permutations could not reverse:
10,000 permutations, then further batches up to 1,000,000 (`docs/audit/remediation/STEP3_PREREG.md` 5.2). A block
still undecided at the cap is `borderline`: it is carried by name and is not a hotspot. Tables written before refs5
have no class column; for them a hotspot is `q_sup_trained < 0.05`. This links the embedding-level signal to the
gene-level DE. The next cell prints the blocks, the analysed and skipped counts and the per-tissue DE totals.

In [2]:
import pandas as pd

from nb_config import DE_DIR
DE_DIR = str(DE_DIR)
de = pd.read_csv(f"{DE_DIR}/de_summary.tsv", sep="\t")

print(f"Total blocks: {len(de)}  |  Hotspots: {de.is_hotspot.sum()}  |  Tissues: {de.tissue.nunique()}")
print("Block status: " + ", ".join(f"{k} {v}" for k, v in de["status"].astype(str).str.split(" ").str[0].value_counts().items()))
print()

# Per-tissue summary
ts = de.groupby("tissue").agg(
    blocks=("cell_type","count"),
    hotspots=("is_hotspot","sum"),
    sig_IHW=("n_sig_dose_IHW","sum"),
    sig_fisher=("n_sig_dose_fisher_BHblock","sum"),
    sig_sex=("n_sig_sex","sum"),
    n_interaction=("n_sig_interaction","sum"),
).reset_index().sort_values("sig_IHW", ascending=False)

ts["hotspot_rate"] = (ts.hotspots / ts.blocks).map(lambda x: f"{x:.0%}")
print(ts[["tissue","blocks","hotspots","hotspot_rate","sig_IHW","sig_fisher","sig_sex","n_interaction"]]
      .rename(columns={"sig_IHW":"DE (IHW)","sig_fisher":"DE (Fisher-BH)",
                        "sig_sex":"DE (sex)","n_interaction":"intxn"})
      .to_string(index=False))

Total blocks: 197  |  Hotspots: 9  |  Tissues: 16
Block status: ok 151, SKIP: 46

tissue  blocks  hotspots hotspot_rate  DE (IHW)  DE (Fisher-BH)  DE (sex)  intxn
 BLOOD      13         4          31%   19358.0         21162.0    9383.0  149.0
  LUNG      21         1           5%    9583.0         10768.0   15280.0 3501.0
   BAT       6         0           0%    7303.0          8312.0    4542.0  195.0
SPLEEN      14         0           0%    6439.0          3881.0   50161.0   23.0
 HEART      16         0           0%    3719.0          3674.0   42226.0   81.0
 SKMVL       9         2          22%    2086.0          2079.0    6319.0   12.0
 SKMGN       9         1          11%    1729.0          1562.0    8792.0    7.0
 WATSC      13         0           0%    1456.0           754.0   89579.0   10.0
 LIVER       8         0           0%     992.0           658.0   41872.0   15.0
SMLINT      13         0           0%     305.0           201.0    6767.0    1.0
KIDNEY      15         0   

In [3]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from nb_config import DE_DIR
de = pd.read_csv(DE_DIR / "de_summary.tsv", sep="\t")

ts = de.groupby("tissue").agg(
    blocks=("cell_type","count"),
    hotspots=("is_hotspot","sum"),
    sig_IHW=("n_sig_dose_IHW","sum"),
    sig_fisher=("n_sig_dose_fisher_BHblock","sum"),
).reset_index().sort_values("sig_IHW", ascending=False)

tissues_ord = ts.tissue.tolist()
colors = plt.cm.tab10(np.linspace(0, 1, len(tissues_ord)))
color_map = dict(zip(tissues_ord, colors))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
fig.suptitle("Stage 10 — Pseudobulk DE Results Overview (limma-trend, IHW FDR)", fontsize=13, fontweight="bold")

# Left: sig_IHW and sig_fisher stacked per tissue
y = np.arange(len(tissues_ord))
ax1.barh(y, ts.sig_IHW,    height=0.7, color=[color_map[t] for t in tissues_ord],
         edgecolor="white", label="IHW (global)")
ax1.barh(y, ts.sig_fisher - ts.sig_IHW, height=0.7,
         left=ts.sig_IHW,
         color=[color_map[t] for t in tissues_ord], edgecolor="white",
         alpha=0.35, label="Fisher-BH only (not IHW)")
for i, (a, b) in enumerate(zip(ts.sig_IHW, ts.sig_fisher)):
    ax1.text(b + 30, i, f"{a:,} / {b:,}", va="center", fontsize=7.5)
ax1.set_yticks(y)
ax1.set_yticklabels(tissues_ord)
ax1.set_xlabel("Number of significant DE genes")
ax1.set_title("Significant Exercise-Response DE Genes\n(IHW global FDR + Fisher sex-combined)")
ax1.legend(fontsize=8, loc="upper right")

# Right: hotspots vs non-hotspots scatter (n_sig_IHW vs Augur AUC)
hot = de[de.is_hotspot]
non = de[~de.is_hotspot]
sc1 = ax2.scatter(non.n_sig_dose_IHW, non.sup_trained_auc,
                  c=[color_map.get(t, "gray") for t in non.tissue],
                  alpha=0.5, s=30, label="non-hotspot")
sc2 = ax2.scatter(hot.n_sig_dose_IHW, hot.sup_trained_auc,
                  c=[color_map.get(t, "black") for t in hot.tissue],
                  edgecolors="black", linewidths=0.8, s=60, marker="*",
                  label="hotspot (gate class)")
ax2.axhline(0.7, color="gray", lw=0.8, ls="--", alpha=0.6, label="AUC=0.7")
ax2.set_xlabel("n_sig_dose_IHW (exercise-DE genes, IHW)")
ax2.set_ylabel("PLS-1 trained AUC (Aim-2 gate)")
ax2.set_title("DE Genes vs Gate AUC\n(stars = hotspot blocks)")
ax2.legend(fontsize=8, loc="upper right")

# Add tissue color legend
from matplotlib.patches import Patch
legend_els = [Patch(color=color_map[t], label=t) for t in tissues_ord]
ax2.legend(handles=legend_els, fontsize=7, ncol=2, loc="upper right")

plt.show()


## Hotspot Blocks

A tissue-by-cell-type block is a hotspot when the Stage 11 gate gives it the class `hotspot`: a
Benjamini-Hochberg call across all analysable blocks on the supervised GeneCompass embedding probe that more
permutations could not reverse (above). The next cell lists the hotspots with
their IHW-significant dose genes; a hotspot can have none (the embedding separates the arms, but the DE on Z finds no
gene that moves with dose). How the list differs from the default before refs5 is in
`docs/audit/remediation/STEP3_RESULTS.md` (B4).

In [4]:
hot_l = de[de.is_hotspot].sort_values(["tissue", "cell_type"])
print(f"{len(hot_l)} hotspots among {int((de['status'] == 'ok').sum())} analysed blocks:")
print(hot_l[["tissue", "cell_type", "sup_trained_auc", "n_sig_dose_IHW"]].to_string(index=False))

9 hotspots among 151 analysed blocks:
tissue                       cell_type  sup_trained_auc  n_sig_dose_IHW
 BLOOD             Classical monocytes            0.821          2500.0
 BLOOD                 Erythroid cells            0.828          2438.0
 BLOOD    Megakaryocytes and platelets            0.837          2572.0
 BLOOD                     Neutrophils            0.819          2239.0
HYPOTH Oligodendrocyte precursor cells            0.798             0.0
  LUNG    Vascular smooth muscle cells            0.817          1391.0
 SKMGN                       Tenocytes            0.825           232.0
 SKMVL                 Satellite cells            0.887           242.0
 SKMVL    Vascular smooth muscle cells            0.874           216.0


In [5]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from nb_config import DE_DIR
de = pd.read_csv(DE_DIR / "de_summary.tsv", sep="\t")
hot = de[de.is_hotspot].sort_values("sup_trained_auc", ascending=True).reset_index(drop=True)

tissue_colors = {
    "BLOOD":"#1f77b4","SKMGN":"#ff7f0e","HYPOTH":"#8c564b",
    "KIDNEY":"#9467bd","LUNG":"#17becf","SKMVL":"#2ca02c",
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8.5), constrained_layout=True)
fig.suptitle(f"Stage 10: Hotspot Blocks ({len(hot)}/{len(de)}; gate class hotspot)",
             fontsize=13, fontweight="bold")

y = np.arange(len(hot))
label = [f"{r.tissue} / {r.cell_type}" for _, r in hot.iterrows()]
bar_colors = [tissue_colors.get(r.tissue, "gray") for _, r in hot.iterrows()]

# Left: Augur AUC
ax1.barh(y, hot.sup_trained_auc, color=bar_colors, edgecolor="white", height=0.7)
ax1.axvline(0.5, color="gray", lw=0.8, ls="--")
ax1.axvline(0.7, color="black", lw=0.8, ls=":", alpha=0.5, label="AUC=0.7")
for i, v in enumerate(hot.sup_trained_auc):
    ax1.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=8)
ax1.set_yticks(y)
ax1.set_yticklabels(label, fontsize=8)
ax1.set_xlabel("PLS-1 trained AUC (trained vs sedentary, v2 embedding)")
ax1.set_title("Hotspot AUC (Aim-2 gate, v2 embedding)")
ax1.set_xlim(0.4, 1.0)
ax1.legend(fontsize=8)

# Right: IHW-significant DE genes
ax2.barh(y, hot.n_sig_dose_IHW, color=bar_colors, edgecolor="white", height=0.7)
for i, v in enumerate(hot.n_sig_dose_IHW):
    ax2.text(v + 5, i, f"{v:,}", va="center", fontsize=8)
ax2.set_yticks(y)
ax2.tick_params(axis="y", labelleft=False)
ax2.set_xlabel("n_sig_dose_IHW (exercise-responsive DE genes)")
ax2.set_title("DE Genes per Hotspot Block (IHW FDR < 0.05)")

# Tissue color legend
from matplotlib.patches import Patch
legend_els = [Patch(color=c, label=t) for t, c in tissue_colors.items()]
ax2.legend(handles=legend_els, fontsize=8, loc="upper right", ncol=2)

plt.show()


## Step 2 — Pre-registered Positive-control Comparison

`compare_posctrl.py` executes the **frozen pre-registration** (`posctrl_prereg.tsv`) against the DE results. Outcomes follow a **miss-ladder** in priority order:

| Outcome | Meaning |
|---------|---------|
| `not_testable_prior` | Gene absent from GeneCompass vocabulary or cell type not resolved |
| `not_testable` | Gene filtered (all-zero) in this tissue × cell type |
| `underpowered` | `mean_fraction < 0.01` or `n_nonzero < 25` |
| `not_significant` | Testable and powered, but FDR ≥ α |
| `wrong_direction` | Significant (FDR < α) but direction is opposite to expectation |
| `recovered` | Significant in the expected direction |
| `recovered_confounded` | Significant but with potential confound |

**Three tiers** of positive controls test different aspects of specificity:

- **Tier A** — direction-anchored: muscle mitochondrial biogenesis genes (up, 8w), heart mitochondrial genes (up), metabolic regulators. Tests that BayesPrism + limma recovers known exercise-responsive biology.
- **Tier Ai** — identity-only: cell-type marker genes (e.g., cardiomyocyte markers in heart) that should be detectable but have no expected direction of exercise response.
- **Tier B** — Vetr et al. named findings: specific gene × tissue × sex × cell-type combinations reported in the MoTrPAC pilot study.
- **Tier C** — Yu immune programs: gene set enrichment of exercise-responsive immune programs in blood cell types.


In [6]:
import pandas as pd
from pathlib import Path

from nb_config import DE_DIR
pr = pd.read_csv(DE_DIR / "posctrl_results.tsv", sep="\t")

print("=== Positive-control outcomes by tier ===")
pivot = pr.groupby(["tier","outcome"]).size().unstack(fill_value=0)
print(pivot.to_string())

print()
print("=== Recovered and wrong_direction entries ===")
notable = pr[pr.outcome.isin(["recovered","recovered_confounded","wrong_direction"])].copy()
notable["lfc_8w"] = notable["lfc_8w"].map(lambda x: f"{x:+.3f}" if pd.notna(x) else "NA")
notable["FDR_IHW"] = notable["FDR_IHW"].map(lambda x: f"{x:.4f}" if pd.notna(x) else "NA")
print(notable[["tier","group","symbol","tissue","cell_type","sex","expected_dir",
               "FDR_IHW","lfc_8w","outcome"]].to_string(index=False))


=== Positive-control outcomes by tier ===
outcome  not_significant  not_testable  not_testable_prior  recovered  wrong_direction
tier                                                                                  
A                     25            10                   1          6                3
Ai                    12             0                   0          2                0
B                      8             5                   2          7                0

=== Recovered and wrong_direction entries ===
tier                  group symbol tissue                        cell_type    sex expected_dir FDR_IHW lfc_8w         outcome
   A muscle_mito_biogenesis Slc2a4  SKMVL            Skeletal muscle cells    any           up  0.0280 +0.199       recovered
   A muscle_mito_biogenesis  Plin4  SKMVL            Skeletal muscle cells    any           up  0.0365 -0.028 wrong_direction
   A muscle_mito_biogenesis   Mfn1  SKMGN            Skeletal muscle cells    any           up  0.

In [7]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from nb_config import DE_DIR
pr = pd.read_csv(DE_DIR / "posctrl_results.tsv", sep="\t")

tier_order   = ["A", "Ai", "B"]
outcome_cats = ["recovered","recovered_confounded","wrong_direction",
                "not_significant","not_testable","not_testable_prior"]
outcome_colors = {
    "recovered":           "#2ca02c",
    "recovered_confounded":"#98df8a",
    "wrong_direction":     "#d62728",
    "not_significant":     "#aec7e8",
    "not_testable":        "#c7c7c7",
    "not_testable_prior":  "#e5e5e5",
}
outcome_labels = {
    "recovered": "Recovered",
    "recovered_confounded": "Recovered (confounded)",
    "wrong_direction": "Wrong direction",
    "not_significant": "Not significant",
    "not_testable": "Not testable (DE filter)",
    "not_testable_prior": "Not testable (vocab/cell type)",
}

counts = pr.groupby(["tier","outcome"]).size().unstack(fill_value=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
fig.suptitle("Stage 10 — Pre-registered Positive-control Comparison", fontsize=13, fontweight="bold")

# Left: stacked bar per tier
tier_totals = []
x = np.arange(len(tier_order))
bottoms = np.zeros(len(tier_order))
for oc in outcome_cats:
    vals = [counts.loc[t, oc] if oc in counts.columns and t in counts.index else 0
            for t in tier_order]
    ax1.bar(x, vals, bottom=bottoms,
            color=outcome_colors[oc], label=outcome_labels[oc], edgecolor="white")
    bottoms += np.array(vals)
for i, t in enumerate(tier_order):
    total = int(bottoms[i])
    rec   = int(counts.loc[t, "recovered"] if "recovered" in counts.columns and t in counts.index else 0)
    rec_c = int(counts.loc[t, "recovered_confounded"] if "recovered_confounded" in counts.columns and t in counts.index else 0)
    ax1.text(i, total + 0.3, f"{rec+rec_c}/{total}", ha="center", fontsize=10, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels([f"Tier {t}" for t in tier_order], fontsize=11)
ax1.set_ylabel("Number of positive controls")
ax1.set_title("Positive-control Recovery by Tier\n(number above bar = recovered/total)")
ax1.legend(fontsize=7.5, bbox_to_anchor=(1.01, 1), loc="upper left")

# Right: FDR vs |LFC| scatter for Tier A + B testable entries
testable = pr[pr.outcome.isin(["recovered","recovered_confounded","wrong_direction","not_significant"])].copy()
testable["abs_lfc"] = testable["lfc_8w"].abs()
testable["neg_log10_fdr"] = (-np.log10(testable["FDR_IHW"].clip(1e-5))).clip(0)
tier_mk = {"A": "o", "Ai": "s", "B": "^"}
tier_col = {"A": "steelblue", "Ai": "orange", "B": "green"}
for tier in ["A", "Ai", "B"]:
    sub = testable[testable.tier == tier]
    ax2.scatter(sub.abs_lfc, sub.neg_log10_fdr,
                marker=tier_mk[tier], color=tier_col[tier],
                s=40, alpha=0.7, label=f"Tier {tier} (n={len(sub)})")
# Highlight recovered
rec = testable[testable.outcome.isin(["recovered","recovered_confounded"])]
ax2.scatter(rec.abs_lfc, rec.neg_log10_fdr, marker="*",
            color="gold", edgecolors="black", s=120, zorder=5, label="Recovered")
ax2.axhline(-np.log10(0.05), color="gray", lw=0.8, ls="--", label="FDR=0.05")
for _, row in rec.iterrows():
    ax2.annotate(row["symbol"], (row["abs_lfc"], row["neg_log10_fdr"]),
                 fontsize=6.5, xytext=(3, 3), textcoords="offset points")
ax2.set_xlabel("|LFC at 8w|")
ax2.set_ylabel("-log10(FDR IHW)")
ax2.set_title("Testable Controls: FDR vs |LFC|\n(★ = recovered)")
ax2.legend(fontsize=7.5)

plt.show()


In [8]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from nb_config import DE_DIR
de  = pd.read_csv(DE_DIR / "de_summary.tsv",      sep="\t")
pc  = pd.read_csv(DE_DIR / "posctrl_responsiveness.tsv", sep="\t")

tissue_colors = {
    "BAT":"#aec7e8","BLOOD":"#1f77b4","CORTEX":"#9467bd","HEART":"#d62728",
    "HIPPOC":"#8c564b","HYPOTH":"#c49c94","KIDNEY":"#e377c2","LIVER":"#7f7f7f",
    "LUNG":"#bcbd22","SKMGN":"#ff7f0e","SKMVL":"#2ca02c","SMLINT":"#98df8a",
    "OVARY":"#ffbb78","SPLEEN":"#9edae5",
    "TESTES":"#c7c7c7","WATSC":"#17becf",
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
fig.suptitle("Stage 10 — Sex-differential Signal & Tier-C Immune Responsiveness", fontsize=13, fontweight="bold")

# Left: n_sig_sex vs n_sig_dose_IHW per block (log scale)
for _, row in de.iterrows():
    col = tissue_colors.get(row.tissue, "gray")
    mk  = "*" if row.is_hotspot else "o"
    ax1.scatter(row.n_sig_dose_IHW + 1, row.n_sig_sex + 1,
                c=col, marker=mk, s=(55 if row.is_hotspot else 20),
                alpha=0.7, edgecolors="black" if row.is_hotspot else "none",
                linewidths=0.5)
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("n_sig_dose_IHW + 1 (exercise DE, log scale)")
ax1.set_ylabel("n_sig_sex + 1 (sex DE, log scale)")
ax1.set_title("Exercise vs Sex DE per Block\n(★ = hotspot)")
from matplotlib.patches import Patch
ax1.legend(handles=[Patch(color=c, label=t) for t, c in tissue_colors.items()],
           fontsize=6.5, ncol=2, loc="upper left")

# Right: Tier C Yu immune programs — enrichment per program
if len(pc) > 0:
    pc_tier_c = pc[pc.tier == "C"].copy() if "tier" in pc.columns else pc.copy()
    pc_agg = pc_tier_c.groupby(["tissue","cell_type"]).agg(
        n_DE=("n_DE","sum"),
        n_testable=("n_testable","sum"),
        enrichment=("enrichment","mean"),
        binom_p=("binom_p","min"),
    ).reset_index()
    sig = pc_agg[pc_agg.binom_p < 0.05].sort_values("enrichment", ascending=False)
    if len(sig) > 0:
        y = np.arange(len(sig))
        cols = [tissue_colors.get(t, "gray") for t in sig.tissue]
        ax2.barh(y, sig.enrichment, color=cols, edgecolor="white", height=0.7)
        for i, (_, row) in enumerate(sig.iterrows()):
            ax2.text(row.enrichment + 0.05, i,
                     f"{row.tissue} / {row.cell_type}  (p={row.binom_p:.3f})",
                     va="center", fontsize=7.5)
        ax2.set_yticks(y)
        ax2.set_yticklabels([f"{r.n_DE}/{r.n_testable}" for _, r in sig.iterrows()], fontsize=8)
        ax2.set_xlabel("Enrichment (program DE rate / background rate)")
        ax2.set_title(f"Tier C — Significant Yu Immune Programs\n(binom_p < 0.05, n={len(sig)})")
        ax2.axvline(1.0, color="gray", lw=0.8, ls="--")
    else:
        ax2.text(0.5, 0.5, "No Tier C programs with binom_p < 0.05",
                 ha="center", va="center", transform=ax2.transAxes)

plt.show()


## Stage Summary

Stage 10 applies a consistent limma-trend framework across tissues, sexes, cell types, and exercise durations.
Results are evaluated against a frozen positive-control set, with global multiplicity correction and explicit
reporting of limited recovery. The next cell computes the headline numbers for this config, among them the
pipeline-sanity subset of the positive controls and the registered check of the DE calls against technical
covariates. refs5 against the default before it is in `docs/audit/remediation/STEP3_RESULTS.md` (B5); the positive
controls are read in `docs/audit/remediation/STEP4_RESULTS.md` (V2).

In [9]:
from IPython.display import Markdown, display
de = pd.read_csv(DE_DIR / "de_summary.tsv", sep="\t")
pr = pd.read_csv(DE_DIR / "posctrl_results.tsv", sep="\t")
ok = de[de["status"] == "ok"]
by_t = ok.groupby("tissue")["n_sig_dose_IHW"].sum().sort_values(ascending=False)
sp = ok[ok.tissue == "SPLEEN"]["n_sig_dose_IHW"]
single = sorted(t for t in de.tissue.unique() if (de[de.tissue == t][["n_male", "n_female"]].min() == 0).any())
n_single = {t: int(ok[ok.tissue == t]["n_sig_dose_IHW"].sum()) for t in single}
tiers = pr.groupby("tier")["outcome"].agg(lambda s: f"{(s == 'recovered').sum()} of {len(s)}")
a_out = pr[pr.tier == "A"]["outcome"].value_counts()
hot = de[de.is_hotspot]
border = de[de["gate_class"] == "borderline"] if "gate_class" in de.columns else de.iloc[0:0]

# distinct genes and the low-fraction tier (de_headline.tsv; written by Stage 10 since refs5)
headline = ""
if (DE_DIR / "de_headline.tsv").is_file():
    h = pd.read_csv(DE_DIR / "de_headline.tsv", sep="\t").set_index("scope").loc["ALL"]
    headline = (f" The pairs are {int(h['n_genes_distinct']):,} distinct genes, and {int(h['n_pairs_not_low_fraction']):,} "
                f"of the pairs lie outside the {int(h['n_blocks_low_fraction'])} analysed blocks whose mean fraction is below 0.01.")

# single-sex tissues: a dose test on the one sex's p since refs5; none in tables written before it
single_tested = "n_dose_tests" in de.columns and ok[ok.tissue.isin(single)]["n_dose_tests"].fillna(0).sum() > 0
if single_tested:
    single_txt = (", ".join(f"{t} has {n:,}" for t, n in n_single.items()) + " dose-significant gene x block pairs. "
                  "A single-sex block's dose p is its one sex's omnibus p (`dose_p` in `de_methods.tsv`).")
else:
    single_txt = (", ".join(single) + f" have {sum(n_single.values())} dose-significant genes by construction: in this "
                  "run the Fisher combination was missing when either sex was absent.")

# the pipeline-sanity subset of the positive controls (POSCTRL_PREREG.md section 4): Tier A in heart and muscle parenchyma
san = pr[(pr.tier == "A") & (((pr.tissue == "HEART") & (pr.cell_type == "Cardiomyocytes"))
                             | (pr.tissue.isin(["SKMGN", "SKMVL"]) & (pr.cell_type == "Skeletal muscle cells")))]
san_rec = int((san.outcome == "recovered").sum())
san_nt = int(san.outcome.isin(["not_testable", "not_testable_prior"]).sum())

# the registered check of the calls against technical covariates (de_covariate_robustness.tsv; refs5 on)
robust_txt = ""
if (DE_DIR / "de_covariate_robustness.tsv").is_file():
    r = pd.read_csv(DE_DIR / "de_covariate_robustness.tsv", sep="\t")
    r = r[r["status_headline"] == "ok"]
    n_head = int(r["n_sig_headline"].sum())
    called = r[r["gate_class"].isin(["hotspot", "borderline"])]
    rob = ((r["retained_share_A"] >= 0.8) & (r["spearman_neglog10p_A"] >= 0.9)
           & (r["retained_share_B"] >= 0.8) & (r["spearman_neglog10p_B"] >= 0.9))
    robust_txt = f"""
- **Technical covariates (registered check).** The whole DE was refitted with RIN and globin percentage in every model
  (A), and with those plus duplication rate and 5'-3' bias (B). Of the {n_head:,} headline pairs,
  {int(r['n_retained_A'].sum()):,} ({r['n_retained_A'].sum() / n_head:.0%}) are retained under A and
  {int(r['n_retained_B'].sum()):,} ({r['n_retained_B'].sum() / n_head:.0%}) under B. A block is ROBUST if at least 80%
  of its calls are retained and the gene ranking correlates at 0.9 or more, under both: {int(rob.sum())} of the
  {len(r)} analysed blocks, and {int(rob[called.index].sum())} of the {len(called)} called blocks. The median
  correlation of the gene ranking with the headline is {r['spearman_neglog10p_A'].median():.2f} under A and
  {r['spearman_neglog10p_B'].median():.2f} under B."""

# sex-consistency at 8 weeks, both readings (columns present since refs5)
sex_txt = ""
if "n_sexcons_concordance_8w" in de.columns:
    sex_txt = f"""
- **Sex-consistent 8-week calls.** {int(ok['n_sexcons_concordance_8w'].sum()):,} by concordance (both per-sex tests
  significant, same direction); {int(ok['n_sexcons_repfdr_8w'].sum()):,} by repfdr on the dose-significant gene-blocks;
  {int(ok['n_sexcons_union_8w'].sum()):,} in their union."""

display(Markdown(f"""
**Results for this config:**

- **Blocks.** {len(de)} blocks, {len(ok)} analysed under the §6 rule, {len(de) - len(ok)} skipped as near-empty;
  {int(ok['n_sig_dose_IHW'].sum()):,} dose-significant gene x block pairs (FDR_IHW < 0.05).{headline} {by_t.index[0]}
  accounts for {int(by_t.iloc[0]):,} of the pairs.""" + (f""" SPLEEN has {int(sp.sum()):,}, spread over its {len(sp)}
  analysable types ({int(sp.min()):,}–{int(sp.max()):,} each). SPLEEN is read differentially only (its estimated
  fractions are implausible).""" if len(sp) else "") + f"""
- **Single-sex tissues.** {single_txt}
- **Positive controls** (frozen spec; `posctrl_prereg_refs4.tsv` renames targets only): recovered in Tier A
  {tiers.get('A', '-')} ({a_out.get('wrong_direction', 0)} wrong direction, {a_out.get('not_significant', 0)} not
  significant, {a_out.get('not_testable', 0) + a_out.get('not_testable_prior', 0)} not testable), Tier Ai
  {tiers.get('Ai', '-')}, Tier B {tiers.get('B', '-')}. **Pipeline-sanity subset** (Tier A in heart cardiomyocytes and
  the two muscles' skeletal muscle cells): {san_rec} of the {len(san) - san_nt} testable controls recovered
  ({san_rec} of {len(san)} in all). The specification reads a miss here as a red flag for the deconvolution-to-DE
  pipeline; the flag stands when fewer than half of the testable controls recover.
- **Hotspots.** The {len(hot)} flagged blocks are the gate's class `hotspot`""" + (f"""; borderline, not a hotspot:
  {", ".join(f"{t} {c}" for t, c in zip(border.tissue, border.cell_type))}""" if len(border) else "") + f""". IHW-significant
  dose genes per hotspot range from {int(hot['n_sig_dose_IHW'].min()):,} to {int(hot['n_sig_dose_IHW'].max()):,}.""" + robust_txt + sex_txt + """
"""))



**Results for this config:**

- **Blocks.** 197 blocks, 151 analysed under the §6 rule, 46 skipped as near-empty;
  53,084 dose-significant genes (FDR_IHW < 0.05). BLOOD accounts for
  19,358 of them. SPLEEN has 6,439, spread over its 12 analysable types
  (461–745 each): the shared bulk signal of deconvolved profiles, not a cell-type-specific
  response. SPLEEN is read differentially only (its estimated fractions are implausible).
- **Single-sex tissues.** OVARY, TESTES have 0
  dose-significant genes by construction: `run_pseudobulk_de.R` combines the two sexes' p values by Fisher's method
  and sets the combined p to missing when either sex is absent. This is structural, not biology.
- **Positive controls** (frozen spec; `posctrl_prereg_refs4.tsv` renames targets only): recovered in Tier A
  6 of 45 (3 wrong direction, 25 not
  significant, 11 not testable), Tier Ai
  2 of 14, Tier B 7 of 22.
- **Hotspots.** The 9 flagged blocks are the gate's. IHW-significant dose genes per hotspot range from
  0 to 2,572.


## Reproducibility Notes

### Production history

The default tree's Stage 10 ran in the refs5 chain job 12377824 (commit `44332e6`, no modified tracked file), after
the gate, whose classes it copies (`deconvolution.hotspots_table`). The same job reran the whole DE with technical
covariates in every model (`pseudobulk_de_covA/`, `pseudobulk_de_covB/`) and compared the calls
(`de_covariate_robustness.tsv`). The results of the default before refs5 (job 12122211), of refs4 on 14 tissues (job
12047418), of v2 on the July references (job 12015440) and April's are superseded.

### Current result checks

The block, hotspot and positive-control counts are computed in the cells above and the checksums in the cell below.
The specification names the Tier A controls in heart and muscle parenchyma as its pipeline-sanity subset and reads a
miss there as a red flag for the deconvolution-to-DE pipeline, not as a biological null
(`deconvolution/POSCTRL_PREREG.md`, section 4). The summary cell above prints that subset's count.

### Re-run Commands

The default config carries v2 with all 16 tissues, so the hotspot table defaults to `deconvolution.hotspots_table`:

```bash
python pipeline/run_stage10.py
python pipeline/run_stage10.py --from 2
```

The frozen file `deconvolution/reference/posctrl_prereg.tsv` (with target names mapped to the refs4 labels in
`posctrl_prereg_refs4.tsv`) defines the gene, tissue, cell-type, sex, and tier specifications. MoTrPAC expression is
used only after model training.

In [10]:
import hashlib, pandas as pd
from pathlib import Path

from nb_config import DE_DIR

print("=== Key output file checksums ===")
for f in ["de_summary.tsv","de_hotspots.tsv","posctrl_results.tsv",
          "posctrl_responsiveness.tsv","posctrl_summary.md"]:
    p = DE_DIR / f
    if p.exists():
        md5 = hashlib.md5(p.read_bytes()).hexdigest()
        n = sum(1 for _ in p.open()) - 1 if f.endswith(".tsv") else "-"
        print(f"  {f:<40}  {md5[:16]}...  ({n} rows)")
    else:
        print(f"  {f:<40}  MISSING")

print()
print("=== Per-tissue DE file counts ===")
from nb_config import TISSUES as tissues
total_files = 0
for t in tissues:
    d = DE_DIR / t
    n = len(list(d.glob("de__*.tsv"))) if d.exists() else 0
    total_files += n
    status = "OK" if n > 0 else "MISSING"
    print(f"  {t:<10}  {n:>3} de__ files  [{status}]")
print(f"  TOTAL: {total_files} per-cell-type DE files")

print()
de = pd.read_csv(DE_DIR / "de_summary.tsv", sep="\t")
print(f"de_summary: {len(de)} blocks, {de.is_hotspot.sum()} hotspots, "
      f"{de.n_sig_dose_IHW.sum():,} total IHW-significant DE genes")


=== Key output file checksums ===
  de_summary.tsv                            dbe26a0628b3bf44...  (197 rows)
  de_hotspots.tsv                           7ee2f1065b971ff5...  (9 rows)
  posctrl_results.tsv                       5dfdef923f6f5572...  (81 rows)
  posctrl_responsiveness.tsv                29bb2295d59c8ad1...  (11 rows)
  posctrl_summary.md                        986729090693be56...  (- rows)

=== Per-tissue DE file counts ===
  BAT           6 de__ files  [OK]
  BLOOD         8 de__ files  [OK]
  CORTEX       10 de__ files  [OK]
  HEART        12 de__ files  [OK]
  HIPPOC        7 de__ files  [OK]
  HYPOTH        8 de__ files  [OK]
  KIDNEY       11 de__ files  [OK]
  LIVER         8 de__ files  [OK]
  LUNG          7 de__ files  [OK]
  OVARY        15 de__ files  [OK]
  SKMGN         8 de__ files  [OK]
  SKMVL         8 de__ files  [OK]
  SMLINT       13 de__ files  [OK]
  SPLEEN       12 de__ files  [OK]
  TESTES        6 de__ files  [OK]
  WATSC        12 de__ files  [O